In [0]:
# Databricks notebook source
# GOLD LAYER: Data Quality Validation
# Gold DQ = business logic correctness + completeness + consistency

from pyspark.sql.functions import col, count, sum as spark_sum, when, lit, current_timestamp

standings   = spark.read.table("f1_data.gold.driver_standings")
team_stand  = spark.read.table("f1_data.gold.team_standings")
race_perf   = spark.read.table("f1_data.gold.race_performance")
season_stat = spark.read.table("f1_data.gold.driver_season_stats")
circuit     = spark.read.table("f1_data.gold.circuit_performance")

print("=" * 70)
print("GOLD LAYER DQ VALIDATION")
print("=" * 70)

# ============ CHECK 1: CHAMPIONSHIP LEADER HAS MOST POINTS ============
leader = spark.sql("""
    SELECT driver_name, championship_position, total_points
    FROM f1_data.gold.driver_standings
    WHERE year = 2024
    ORDER BY championship_position
    LIMIT 1
""").collect()[0]

all_pts = spark.sql("""
    SELECT MAX(total_points) AS max_pts
    FROM f1_data.gold.driver_standings
    WHERE year = 2024
""").collect()[0]["max_pts"]

p1_is_leader = leader["total_points"] == all_pts
print(f"\n✅ CHECK 1: Championship leader has max points")
print(f"   Leader: {leader['driver_name']} — {leader['total_points']} pts")
print(f"   {'PASS' if p1_is_leader else 'FAIL ⚠️'}")

# ============ CHECK 2: NO NEGATIVE POINTS IN ANY GOLD TABLE ============
neg_driver   = standings.filter(col("total_points") < 0).count()
neg_team     = team_stand.filter(col("total_points") < 0).count()
neg_race     = race_perf.filter(col("points") < 0).count()
neg_season   = season_stat.filter(col("total_points") < 0).count()

print(f"\n✅ CHECK 2: No negative points")
print(f"   driver_standings: {neg_driver} negative {'PASS' if neg_driver == 0 else 'FAIL ⚠️'}")
print(f"   team_standings:   {neg_team} negative {'PASS' if neg_team == 0 else 'FAIL ⚠️'}")
print(f"   race_performance: {neg_race} negative {'PASS' if neg_race == 0 else 'FAIL ⚠️'}")
print(f"   driver_season_stats: {neg_season} negative {'PASS' if neg_season == 0 else 'FAIL ⚠️'}")

# ============ CHECK 3: POINTS CONSISTENCY ACROSS GOLD TABLES ============
# Total points in driver_standings should match driver_season_stats
standings_pts = spark.sql("""
    SELECT driver_number, total_points AS pts_standings
    FROM f1_data.gold.driver_standings WHERE year = 2024
""")

season_pts = spark.sql("""
    SELECT driver_number, total_points AS pts_season
    FROM f1_data.gold.driver_season_stats WHERE year = 2024
""")

mismatch = standings_pts.join(season_pts, "driver_number") \
    .filter(col("pts_standings") != col("pts_season"))

print(f"\n✅ CHECK 3: Points consistent across Gold tables")
print(f"   Mismatches between standings and season stats: {mismatch.count()}")
print(f"   {'PASS' if mismatch.count() == 0 else 'FAIL ⚠️'}")

if mismatch.count() > 0:
    mismatch.show()

# ============ CHECK 4: ALL DRIVERS IN STANDINGS ============
drivers_in_standings  = standings.select("driver_number").distinct().count()
drivers_in_season     = season_stat.select("driver_number").distinct().count()

print(f"\n✅ CHECK 4: Driver coverage")
print(f"   Drivers in standings:    {drivers_in_standings}")
print(f"   Drivers in season stats: {drivers_in_season}")
print(f"   {'PASS' if drivers_in_standings == drivers_in_season else 'REVIEW ⚠️ (may be OK if sprint-only drivers differ)'}")

# ============ CHECK 5: WIN COUNT ACCURACY ============
# Driver with most wins should have 1st place in standings
most_wins_driver = spark.sql("""
    SELECT driver_name, total_wins, total_points, championship_position
    FROM f1_data.gold.driver_standings
    WHERE year = 2024
    ORDER BY total_wins DESC
    LIMIT 3
""")

print(f"\n✅ CHECK 5: Top winners sanity check")
most_wins_driver.show(truncate=False)

# ============ CHECK 6: RATE METRICS IN VALID RANGE ============
invalid_rates = season_stat.filter(
    (col("win_rate") < 0) | (col("win_rate") > 100) |
    (col("podium_rate") < 0) | (col("podium_rate") > 100) |
    (col("reliability_score") < 0) | (col("reliability_score") > 100)
).count()

print(f"\n✅ CHECK 6: Rate metrics in valid range (0-100)")
print(f"   Invalid rate records: {invalid_rates}")
print(f"   {'PASS' if invalid_rates == 0 else 'FAIL ⚠️'}")

# ============ WRITE DQ LOG ============
gold_dq = [
    ("driver_standings", "leader_has_max_points", "total_points", str(leader['total_points']), str(all_pts), p1_is_leader),
    ("driver_standings", "no_negative_points", "total_points", str(neg_driver), "0", neg_driver == 0),
    ("team_standings", "no_negative_points", "total_points", str(neg_team), "0", neg_team == 0),
    ("gold_tables", "points_consistency", "total_points", str(mismatch.count()), "0", mismatch.count() == 0),
    ("driver_season_stats", "rate_metrics_valid", "win_rate", str(invalid_rates), "0", invalid_rates == 0),
]

gold_dq_df = spark.createDataFrame(
    gold_dq, ["table_name", "check_type", "column", "value", "threshold", "pass"]
).withColumn("timestamp", current_timestamp())

gold_dq_df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable(
    "f1_data.gold._data_quality_log"
)

total_failures = sum(1 for row in gold_dq if not row[5])
if total_failures > 0:
    raise Exception(f"{total_failures} Gold DQ checks FAILED — dashboard will show incorrect data")
else:
    print(f"\n✅ ALL GOLD DQ CHECKS PASSED")

GOLD LAYER DQ VALIDATION

✅ CHECK 1: Championship leader has max points
   Leader: Max Verstappen — 331.0 pts
   PASS

✅ CHECK 2: No negative points
   driver_standings: 0 negative PASS
   team_standings:   0 negative PASS
   race_performance: 0 negative PASS
   driver_season_stats: 0 negative PASS

✅ CHECK 3: Points consistent across Gold tables
   Mismatches between standings and season stats: 0
   PASS

✅ CHECK 4: Driver coverage
   Drivers in standings:    23
   Drivers in season stats: 23
   PASS

✅ CHECK 5: Top winners sanity check
+---------------+----------+------------+---------------------+
|driver_name    |total_wins|total_points|championship_position|
+---------------+----------+------------+---------------------+
|Max Verstappen |10        |331.0       |1                    |
|Lando Norris   |3         |279.0       |2                    |
|Charles Leclerc|2         |245.0       |3                    |
+---------------+----------+------------+---------------------+


✅ CHEC